# Image Forgery Detection — Dual-Input EfficientNetB3

Two EfficientNetB3 backbones (raw RGB + ELA residuals) fused via concatenation.

**Run time:** ~15-25 min on Colab T4 GPU (first run includes cache precomputation)

In [ ]:
# === COLAB SETUP ===
# Set Runtime → Change runtime type → T4 GPU → Run all

import os
IN_COLAB = 'COLAB_GPU' in os.environ

if IN_COLAB:
    import subprocess, sys, importlib
    _pkgs = {'scikit-learn': 'sklearn', 'Pillow': 'PIL', 'opencv-python-headless': 'cv2', 'kagglehub': 'kagglehub'}
    for pkg, mod in _pkgs.items():
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    from pathlib import Path
    import kagglehub
    try:
        DATASET_DIR = kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")
        for child in Path(DATASET_DIR).rglob('*'):
            if child.name == 'Au' and child.is_dir():
                DATASET_DIR = str(child.parent)
                break
        print(f'Dataset: {DATASET_DIR}')
    except Exception as e:
        print(f'Kaggle download failed: {e}\nSet DATASET_DIR manually above if needed.')
        DATASET_DIR = None
else:
    DATASET_DIR = None

print('Setup complete.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import json, tempfile, gc
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
import cv2
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Detect dataset
if DATASET_DIR:
    DATA_ROOT = Path(DATASET_DIR)
else:
    # Local development fallback paths — adjust for your setup
    candidates = [Path.cwd() / 'dataset', Path('C:/Users/sange/Downloads/nikunj_image_f/dataset')]
    DATA_ROOT = next((p for p in candidates if p.exists()), None)

assert DATA_ROOT and DATA_ROOT.exists(), f'Dataset not found. Set DATASET_DIR above.'
assert (DATA_ROOT / 'Au').exists() and (DATA_ROOT / 'Tp').exists(), 'Missing Au/ or Tp/ folders'
print(f'Dataset: {DATA_ROOT}')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
ELA_QUALITY = 91

In [ ]:
def compute_ela(image_path, quality=ELA_QUALITY, target_size=IMG_SIZE, pil_image=None):
    img = pil_image if pil_image is not None else Image.open(image_path).convert('RGB')
    fd, tmp = tempfile.mkstemp(suffix='.jpg')
    try:
        os.close(fd)
        img.save(tmp, 'JPEG', quality=quality)
        compressed = Image.open(tmp)
        ela = ImageChops.difference(img, compressed)
        extrema = ela.getextrema()
        max_diff = max(ex[1] for ex in extrema)
        if max_diff == 0: max_diff = 1
        ela = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
        return ela.resize(target_size, Image.LANCZOS)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def collect_paths(d):
    return sorted([str(p) for p in Path(d).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

au_paths = collect_paths(DATA_ROOT / 'Au')
tp_paths = collect_paths(DATA_ROOT / 'Tp')
n = min(len(au_paths), len(tp_paths))
random.seed(SEED)
au_paths = random.sample(au_paths, n)
tp_paths = random.sample(tp_paths, n)
print(f'Balanced: {n} authentic, {n} tampered')

all_paths = au_paths + tp_paths
all_labels = [1]*n + [0]*n

In [ ]:
# 70/15/15 stratified split (paths only — no images loaded yet)
paths_tmp, paths_test, labels_tmp, y_test = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=SEED, stratify=all_labels)
paths_train, paths_val, y_train, y_val = train_test_split(
    paths_tmp, labels_tmp, test_size=0.15/(1-0.15), random_state=SEED, stratify=labels_tmp)
y_val = np.array(y_val, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)
y_train = np.array(y_train, dtype=np.int32)
del all_paths, all_labels, paths_tmp, labels_tmp
print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')
print(f'Train balance: {sum(y_train)}/{len(y_train)} (1s/{len(y_train)})')

In [ ]:
# Precompute resized JPEG + ELA PNG cache for all splits
# Eliminates py_function dependency and ELA encoder mismatch
CACHE_DIR = DATA_ROOT / 'cache'

def prepare_cache(paths, desc):
    raw_cache, ela_cache = [], []
    for i, p in enumerate(paths):
        rel = Path(p).relative_to(DATA_ROOT)
        rp = CACHE_DIR / 'raw' / rel.with_suffix('.jpg')
        ep = CACHE_DIR / 'ela' / rel.with_suffix('.png')
        raw_cache.append(str(rp))
        ela_cache.append(str(ep))
        if not rp.exists() or not ep.exists():
            img = Image.open(p).convert('RGB')
            rp.parent.mkdir(parents=True, exist_ok=True)
            img.resize(IMG_SIZE, Image.LANCZOS).save(rp, 'JPEG', quality=95)
            ep.parent.mkdir(parents=True, exist_ok=True)
            compute_ela(p, quality=ELA_QUALITY, target_size=IMG_SIZE, pil_image=img).save(ep)
        if (i+1) % 1000 == 0: print(f'  {desc}: {i+1}/{len(paths)}')
    return raw_cache, ela_cache

print('Precomputing cache...')
raw_train, ela_train = prepare_cache(paths_train, 'train')
raw_val, ela_val = prepare_cache(paths_val, 'val')
raw_test, ela_test = prepare_cache(paths_test, 'test')
print('Cache ready.')
gc.collect()

In [ ]:
def load_image(jpeg_path, png_path):
    raw = tf.image.decode_jpeg(tf.io.read_file(jpeg_path), channels=3)
    ela = tf.image.decode_png(tf.io.read_file(png_path), channels=3)
    raw.set_shape([*IMG_SIZE, 3])
    ela.set_shape([*IMG_SIZE, 3])
    return raw, ela

def make_dual_ds(raw_paths, ela_paths, labels, training=False):
    rot_layer = tf.keras.layers.RandomRotation(25/360, fill_mode='nearest', seed=SEED) if training else None
    zoom_layer = tf.keras.layers.RandomZoom(height_factor=(-0.1, 0.1), fill_mode='nearest', seed=SEED) if training else None
    ds = tf.data.Dataset.from_tensor_slices((raw_paths, ela_paths, labels))
    def _process(rp, ep, label):
        raw, ela = load_image(rp, ep)
        if training:
            raw = tf.image.random_brightness(raw, 0.15)
            raw = tf.clip_by_value(tf.cast(raw, tf.int32), 0, 255)
            raw = tf.image.random_contrast(raw, 0.7, 1.3)
            raw = tf.clip_by_value(tf.cast(raw, tf.int32), 0, 255)
            raw_u8 = tf.cast(raw, tf.uint8)
            raw_u8 = tf.image.random_hue(raw_u8, 0.05)
            raw_u8 = tf.image.random_saturation(raw_u8, 0.8, 1.2)
            raw_ela = tf.stack([raw_u8, ela], axis=0)
            raw_ela = tf.image.random_flip_left_right(raw_ela, seed=SEED)
            raw_ela = rot_layer(raw_ela)
            raw_ela = zoom_layer(raw_ela)
            raw, ela = tf.cast(raw_ela[0], tf.int32), tf.cast(raw_ela[1], tf.int32)
        raw = preprocess_input(tf.cast(raw, tf.float32))
        ela = preprocess_input(tf.cast(ela, tf.float32))
        return {'raw_input': raw, 'ela_input': ela}, tf.cast(label, tf.float32)
    if training:
        ds = ds.shuffle(min(len(labels), 4096), seed=SEED)
    return ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(3)

train_ds = make_dual_ds(raw_train, ela_train, y_train, training=True)
val_ds = make_dual_ds(raw_val, ela_val, y_val)
test_ds = make_dual_ds(raw_test, ela_test, y_test)
del raw_train, ela_train, raw_val, ela_val, raw_test, ela_test
print('Datasets ready.')

In [ ]:
# Dual-input EfficientNetB3
raw_in = layers.Input(shape=(224,224,3), name='raw_input')
ela_in = layers.Input(shape=(224,224,3), name='ela_input')

# Download weights once, load into two uniquely-named backbones
BASE_WT = 'https://storage.googleapis.com/keras-applications/efficientnetb3_notop.h5'
wt_path = tf.keras.utils.get_file('efficientnetb3_notop.h5', BASE_WT)

raw_bb = EfficientNetB3(include_top=False, weights=None, input_shape=(224,224,3), name='efficientnetb3_raw')
ela_bb = EfficientNetB3(include_top=False, weights=None, input_shape=(224,224,3), name='efficientnetb3_ela')
raw_bb.load_weights(wt_path)
ela_bb.load_weights(wt_path)
raw_bb.trainable = False
ela_bb.trainable = False

raw_feat = layers.GlobalAveragePooling2D()(raw_bb(raw_in))
ela_feat = layers.GlobalAveragePooling2D()(ela_bb(ela_in))

fused = layers.Concatenate()([raw_feat, ela_feat])
fused = layers.Dropout(0.3)(fused)
fused = layers.BatchNormalization()(fused)
fused = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(fused)
fused = layers.Dropout(0.6)(fused)
fused = layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(fused)
fused = layers.Dropout(0.5)(fused)
out = layers.Dense(1, activation='sigmoid', name='forgery_prob')(fused)

model = Model(inputs=[raw_in, ela_in], outputs=out)
model.summary()

In [ ]:
def calibrate_threshold(p_auth, y_true):
    best = {'thr': 0.5, 'score': -1, 'tampered_f1': 0, 'authentic_f1': 0}
    for thr in np.linspace(0.05, 0.95, 181):
        yp = (p_auth >= thr).astype(int)
        tp_t = np.sum((y_true==0)&(yp==0)); fp_t = np.sum((y_true==1)&(yp==0)); fn_t = np.sum((y_true==0)&(yp==1))
        f1_t = 2*tp_t/(2*tp_t+fp_t+fn_t+1e-12)
        tp_a = np.sum((y_true==1)&(yp==1)); fp_a = np.sum((y_true==0)&(yp==1)); fn_a = np.sum((y_true==1)&(yp==0))
        f1_a = 2*tp_a/(2*tp_a+fp_a+fn_a+1e-12)
        score = 0.6*f1_t + 0.4*f1_a - 0.05*abs(thr-0.5)
        if score > best['score']:
            best = {'thr': float(thr), 'score': float(score), 'tampered_f1': float(f1_t), 'authentic_f1': float(f1_a)}
    return best

In [ ]:
# Phase 1: frozen backbones
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

cbs = [
    EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_phase1_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
]
print('PHASE 1: Frozen backbones')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs, verbose=1)

In [ ]:
# Phase 2: fine-tune last 12 layers + Focal Loss + SWA
def focal_loss(gamma=2., alpha=0.5):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(alpha_t * (1 - pt)**gamma * ce)
    return loss

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch=12):
        super().__init__()
        self.start_epoch = start_epoch
        self.avg_weights = None
        self.n = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch + 1 >= self.start_epoch:
            w = self.model.get_weights()
            if self.avg_weights is None:
                self.avg_weights = [x.copy() for x in w]
            else:
                for a, x in zip(self.avg_weights, w):
                    a[:] = a * self.n / (self.n + 1) + x / (self.n + 1)
            self.n += 1
    def on_train_end(self, logs=None):
        if self.avg_weights is not None:
            self.model.set_weights(self.avg_weights)
            print(f'SWA: averaged {self.n} checkpoints')

EPOCHS_P2 = 20
steps_per_epoch = len(y_train) // BATCH_SIZE
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-6,
    decay_steps=EPOCHS_P2 * steps_per_epoch,
    alpha=0.02
)
for bb in [raw_bb, ela_bb]:
    bb.trainable = True
    for i, l in enumerate(bb.layers):
        l.trainable = i >= len(bb.layers) - 12

model.compile(optimizer=tf.keras.optimizers.AdamW(
                  learning_rate=lr_schedule,
                  weight_decay=1e-4,
                  global_clipnorm=1.0
              ),
              loss=focal_loss(gamma=2., alpha=0.5),
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

cbs2 = [
    EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_forgery_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
    SWACallback(start_epoch=12),
]
print('PHASE 2: Fine-tuning')
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P2, callbacks=cbs2, verbose=1)

In [ ]:
# Evaluate + save
val_preds = model.predict(val_ds, verbose=0).ravel()
best = calibrate_threshold(val_preds, y_val)

test_preds = model.predict(test_ds, verbose=0).ravel()
y_pred = (test_preds >= best['thr']).astype(int)

acc = float(np.mean(y_pred == y_test))
tamp = float(np.mean(y_pred[y_test==0]==0)) if np.any(y_test==0) else 0
auth = float(np.mean(y_pred[y_test==1]==1)) if np.any(y_test==1) else 0
cm = confusion_matrix(y_test, y_pred, labels=[0,1])

model.save('best_forgery_model.keras')
with open('best_threshold.json', 'w') as f:
    json.dump({'threshold_authentic': best['thr']}, f)

print(f'\n=== RESULTS ===')
print(f'Threshold: {best["thr"]:.3f}')
print(f'Test Accuracy: {acc*100:.2f}%')
print(f'Tampered: {tamp*100:.2f}% | Authentic: {auth*100:.2f}%')
print(f'Tampered F1: {best["tampered_f1"]:.4f} | Authentic F1: {best["authentic_f1"]:.4f}')
print(f'\nConfusion Matrix [rows=true, cols=pred]:\n{cm}')

In [ ]:
# Grad-CAM demo (loads 2 test images on-demand)
# Build Grad-CAM helper model once (reused for both images)
raw_sub = model.get_layer('efficientnetb3_raw')
conv_name = next((l.name for l in reversed(raw_sub.layers) if isinstance(l, layers.Conv2D)), None)
assert conv_name is not None, f'No Conv2D layer in {raw_sub.name}'
gm = Model(model.inputs, [raw_sub.get_layer(conv_name).output, model.output])

def gradcam(raw01, ela01):
    r = tf.convert_to_tensor(raw01[None,...])
    e = tf.convert_to_tensor(ela01[None,...])
    with tf.GradientTape() as tape:
        co, pr = gm([r, e], training=False)
        s = pr[:,0]
    g = tape.gradient(s, co)
    cam = tf.reduce_sum(co[0] * tf.reduce_mean(g, axis=(0,1,2)), axis=-1)
    cam = tf.maximum(cam, 0)
    return (cam / (tf.reduce_max(cam)+1e-8)).numpy()

def overlay(img01, cam, a=0.4):
    h, w = img01.shape[:2]
    c = cv2.applyColorMap(np.uint8(255*cv2.resize(cam,(w,h))), cv2.COLORMAP_JET)[:,:,::-1]/255.0
    return np.clip((1-a)*img01 + a*c, 0, 1)

# Load 2 test images from paths
gc_paths = [paths_test[0], paths_test[len(paths_test)//2]]
gc_ys = [y_test[0], y_test[len(y_test)//2]]
raw_01, ela_01 = [], []
for p in gc_paths:
    img = Image.open(p).convert('RGB').resize(IMG_SIZE, Image.LANCZOS)
    raw_01.append(np.asarray(img, dtype=np.float32) / 255.0)
    ela_01.append(np.asarray(compute_ela(p, quality=ELA_QUALITY, target_size=IMG_SIZE), dtype=np.float32) / 255.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(2):
    r, e = raw_01[i], ela_01[i]
    p = float(model.predict_on_batch([r[None,...], e[None,...]])[0,0])
    lbl = 'Auth' if p >= best['thr'] else 'Fake'
    true = 'Auth' if gc_ys[i]==1 else 'Fake'
    cam = gradcam(preprocess_input(r*255), preprocess_input(e*255))
    axes[i,0].imshow(r); axes[i,0].set_title(f'Raw ({true})')
    axes[i,1].imshow(e); axes[i,1].set_title('ELA')
    axes[i,2].imshow(overlay(r, cam)); axes[i,2].set_title(f'{lbl} {p:.3f}')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.show()
del raw_01, ela_01, gc_paths, gc_ys  # free memory

In [ ]:
# Copy model to Drive (optional — run this manually after training)
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        import shutil
        for f in ['best_forgery_model.keras', 'best_phase1_model.keras', 'best_threshold.json']:
            if Path(f).exists():
                shutil.copy(f, '/content/drive/MyDrive/')
                print(f'Copied {f} to Drive')
    except Exception as e:
        print(f'Drive copy failed (non-fatal): {e}')